# NOFE Voxel-Level Clustering — Data-Driven Brain Parcellation

For a single subject, extract NOFE hidden states (32-dim) for thousands of brain voxels,
then apply **K-Means** to cluster voxels by their learned microstructural signature.

**Question**: Do the clusters correspond to known anatomical regions?

Pipeline:
```
dMRI (143-dim per voxel) → NOFE GNN → hidden states (32-dim)
→ aggregate per coordinate → K-Means → 3D brain parcellation
```

In [1]:
import sys, json, gc
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import nibabel as nib
import torch
import torch.nn.functional as F
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

project_root = Path('../../').resolve()
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(project_root / 'dMRI' / 'scripts'))

from utilities import KernelNN
from utilities_dMRI import dMRI_Graph_Generator

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

using Device: mps
Device: mps


## 1. Load NOFE Model

In [2]:
model_type   = 'regular'
model_prefix = 'base'

trained_models_path = project_root / 'dMRI' / 'trained_models' / model_type
available = sorted([f.stem for f in trained_models_path.glob(f'{model_prefix}_*.pth')])
model_id  = available[-1]
print(f'Model: {model_id}')

with open(trained_models_path / 'logs' / 'configs' / f'{model_id}.json') as f:
    config = json.load(f)

model = KernelNN(
    width=config['width'], ker_width=config['ker_width'], depth=config['depth'],
    in_width=config['node_features'], ker_in=config['edge_attr'], out_width=config['out_width'],
)
model.load_state_dict(torch.load(trained_models_path / f'{model_id}.pth', map_location='cpu'))
model.eval().to(device)

EMBED_DIM = config['width']  # 32
print(f'Embedding dim: {EMBED_DIM}')

@torch.no_grad()
def extract_nofe(graph):
    x  = model.fc1(graph.x.to(device))
    for _ in range(model.depth):
        x = F.relu(model.conv1(x, graph.edge_index.to(device), graph.edge_attr.to(device)))
    return x.cpu().numpy()  # (N, 32)

Model: base_2026-04-29_t11-16-23
Embedding dim: 32


## 2. Extract Voxel Hidden States

Sample large graphs repeatedly and aggregate hidden states per coordinate grid cell.
`GRID=80` means each cell spans ~1.3% of normalized brain space (~1.7mm at 1.05mm resolution).

In [ ]:
data_root    = project_root / 'dMRI' / 'data'
DIFF_SUBPATH = 'T1w/Diffusion_7T/'

all_subjects   = sorted([d for d in data_root.iterdir() if d.is_dir() and d.name.isdigit()])
local_subjects = all_subjects[:10]          # 最多取前 10 个
print(f'Using {len(local_subjects)} subjects: {[d.name for d in local_subjects]}')

SUBJECT = local_subjects[0]
path    = str(SUBJECT / DIFF_SUBPATH) + '/'
print(f'\nMain subject: {SUBJECT.name}')

mask_img = nib.load(path + 'nodif_brain_mask.nii.gz')
mask_arr = mask_img.get_fdata().astype(bool)
VOL_SHAPE = mask_arr.shape
print(f'Volume shape: {VOL_SHAPE}  |  Brain voxels: {mask_arr.sum():,}')

Generator = dMRI_Graph_Generator(str(data_root))
Generator.load_data(path=path)

GRID      = 80
N_SAMPLE  = 6000
N_REPEATS = 3

def _flush_mps():
    if device.type == 'mps':
        torch.mps.empty_cache()

coord_hidden = defaultdict(list)

for rep in range(N_REPEATS):
    print(f'  rep {rep+1}/{N_REPEATS} ...', end=' ', flush=True)
    graph  = Generator.sample_graph(
        N=N_SAMPLE, k=config['k'], perplexity=config['perplexity'],
        sampling_method='random', gen_feature_mode=config['feature_mode'],
        sigma_method=config['sigma_method'],
    )
    hidden = extract_nofe(graph)
    coords = graph.normed_coords.cpu().numpy()
    keys   = (coords * GRID).round().astype(int)
    for h, k in zip(hidden, keys):
        coord_hidden[tuple(k)].append(h)
    del graph, hidden, coords, keys
    _flush_mps()
    print(f'{len(coord_hidden):,} unique super-voxels')
    gc.collect()

all_keys   = list(coord_hidden.keys())
all_coords = np.stack([np.array(k) / GRID for k in all_keys])
all_hidden = np.stack([np.mean(v, axis=0) for v in coord_hidden.values()])
print(f'\nAggregated super-voxels: {len(all_coords):,}')
print(f'Hidden state shape     : {all_hidden.shape}')

Using 10 subjects: ['115017', '115825', '116726', '118225', '125525', '126426', '128935', '130518', '131217', '131722']

Main subject: 115017
Volume shape: (173, 207, 173)  |  Brain voxels: 1,225,932
  rep 1/3 ... 

/Users/siche9897/Github/NOFE/src/generator_utility.py:532: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  normed_coords = torch.tensor(normed_coords, device=device, dtype=torch.float32)


5,917 unique super-voxels
  rep 2/3 ... 11,674 unique super-voxels
  rep 3/3 ... 

/Users/siche9897/Github/NOFE/src/generator_utility.py:532: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  normed_coords = torch.tensor(normed_coords, device=device, dtype=torch.float32)


17,261 unique super-voxels


/Users/siche9897/Github/NOFE/src/generator_utility.py:532: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  normed_coords = torch.tensor(normed_coords, device=device, dtype=torch.float32)



Aggregated super-voxels: 17,261
Hidden state shape     : (17261, 32)


: 

## 3. Elbow Curve + Silhouette Score

In [ ]:
X_clust = StandardScaler().fit_transform(all_hidden)

K_range  = range(2, 9)   # K=2..8 已足够
inertias = []
sils     = []

for k in K_range:
    km  = KMeans(n_clusters=k, random_state=0, n_init=10)
    lbl = km.fit_predict(X_clust)
    inertias.append(km.inertia_)
    # sample_size=2000 → pairwise matrix 2000×2000×8B = 32MB，安全
    sil = silhouette_score(X_clust, lbl, sample_size=2000, random_state=0)
    sils.append(sil)
    print(f'  K={k}  inertia={km.inertia_:,.0f}  silhouette={sil:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(K_range), inertias, 'o-', color='steelblue')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia'); axes[0].set_title('Elbow curve')
axes[0].grid(alpha=0.3)
axes[1].plot(list(K_range), sils, 'o-', color='darkorange')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette score'); axes[1].set_title('Silhouette score')
axes[1].grid(alpha=0.3)
best_k = list(K_range)[np.argmax(sils)]
axes[1].axvline(best_k, color='red', ls='--', label=f'Best K={best_k}')
axes[1].legend()
plt.tight_layout(); plt.show()
print(f'Best K by silhouette: {best_k}')

## 4. Final K-Means Clustering

In [ ]:
N_CLUSTERS = best_k  # or set manually, e.g. N_CLUSTERS = 5

km = KMeans(n_clusters=N_CLUSTERS, random_state=0, n_init=20)
cluster_labels = km.fit_predict(X_clust)

for c in range(N_CLUSTERS):
    n = (cluster_labels == c).sum()
    center = all_coords[cluster_labels == c].mean(axis=0)
    print(f'Cluster {c}: {n:5,} voxels  center=(y={center[1]:.2f}, z={center[2]:.2f})')

## 5. 3D Scatter Visualization (Axial / Coronal / Sagittal)

In [ ]:
CMAP  = plt.cm.get_cmap('tab10', N_CLUSTERS)
colors = CMAP(cluster_labels)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor='white')
planes = [
    ('Axial (X–Y)',    0, 1, 'X (R→L)', 'Y (P→A)'),
    ('Coronal (X–Z)',  0, 2, 'X (R→L)', 'Z (I→S)'),
    ('Sagittal (Y–Z)', 1, 2, 'Y (P→A)', 'Z (I→S)'),
]

for ax, (title, xi, yi, xl, yl) in zip(axes, planes):
    sc = ax.scatter(
        all_coords[:, xi], all_coords[:, yi],
        c=cluster_labels, cmap='tab10', vmin=0, vmax=N_CLUSTERS - 1,
        s=2, alpha=0.5, linewidths=0,
    )
    ax.set_xlabel(xl, fontsize=10); ax.set_ylabel(yl, fontsize=10)
    ax.set_title(title, fontsize=12)
    ax.set_aspect('equal')
    ax.set_facecolor('#111111')

cb = plt.colorbar(sc, ax=axes[-1], ticks=range(N_CLUSTERS), fraction=0.03)
cb.set_label('Cluster')
fig.suptitle(f'NOFE Voxel Clustering (K={N_CLUSTERS}) — Subject {SUBJECT.name}',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 6. 3D Brain Slice View

In [ ]:
# Reconstruct 3D label volume
label_vol = np.zeros(VOL_SHAPE, dtype=np.int8)  # 0 = background
for coord, lbl in zip(all_coords, cluster_labels):
    ijk = (coord * (np.array(VOL_SHAPE) - 1)).round().astype(int)
    ijk = np.clip(ijk, 0, np.array(VOL_SHAPE) - 1)
    label_vol[ijk[0], ijk[1], ijk[2]] = lbl + 1  # 1-indexed; 0=background

bounds = np.array([0] + [i + 1 for i in range(N_CLUSTERS)])
norm   = mcolors.BoundaryNorm(bounds, CMAP.N)

N_SLICES = 5
fig, axes = plt.subplots(3, N_SLICES, figsize=(N_SLICES * 3.5, 9), facecolor='black')

z_idx = np.linspace(int(VOL_SHAPE[2]*0.25), int(VOL_SHAPE[2]*0.75), N_SLICES).astype(int)
y_idx = np.linspace(int(VOL_SHAPE[1]*0.25), int(VOL_SHAPE[1]*0.75), N_SLICES).astype(int)
x_idx = np.linspace(int(VOL_SHAPE[0]*0.25), int(VOL_SHAPE[0]*0.75), N_SLICES).astype(int)

for j, z in enumerate(z_idx):
    slc = label_vol[:, :, z]
    axes[0, j].imshow(slc.T, origin='lower', cmap=CMAP, norm=norm)
    axes[0, j].set_title(f'Axial z={z}', color='white', fontsize=9)
    axes[0, j].axis('off')

for j, y in enumerate(y_idx):
    slc = label_vol[:, y, :]
    axes[1, j].imshow(slc.T, origin='lower', cmap=CMAP, norm=norm)
    axes[1, j].set_title(f'Coronal y={y}', color='white', fontsize=9)
    axes[1, j].axis('off')

for j, x in enumerate(x_idx):
    slc = label_vol[x, :, :]
    axes[2, j].imshow(slc.T, origin='lower', cmap=CMAP, norm=norm)
    axes[2, j].set_title(f'Sagittal x={x}', color='white', fontsize=9)
    axes[2, j].axis('off')

fig.suptitle(f'NOFE Parcellation (K={N_CLUSTERS}) — Subject {SUBJECT.name}',
             fontsize=13, color='white', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Compare with Anatomical Region Masks

Check if NOFE clusters align with the coordinate-based anatomical regions.

In [ ]:
def region_masks(coords):
    y, z = coords[:, 1], coords[:, 2]
    return {
        'frontal':     y > 0.60,
        'parietal':    (y > 0.35) & (y < 0.85) & (z > 0.62),
        'temporal':    (y > 0.20) & (y < 0.70) & (z < 0.55),
        'occipital':   y < 0.25,
        'subcortical': (y > 0.35) & (y < 0.65) & (z > 0.28) & (z < 0.58),
    }

masks = region_masks(all_coords)

# Cluster composition per anatomical region
print(f'{'Region':12s}  ' + '  '.join(f'Clust{c}' for c in range(N_CLUSTERS)))
print('-' * (12 + 9 * N_CLUSTERS))
for rname, rmask in masks.items():
    row = f'{rname:12s}'
    for c in range(N_CLUSTERS):
        n = ((cluster_labels == c) & rmask).sum()
        row += f'  {n:6d}'
    print(row)

# Heatmap: anatomical region × cluster
region_names = list(masks.keys())
mat = np.zeros((len(region_names), N_CLUSTERS))
for i, (rname, rmask) in enumerate(masks.items()):
    for c in range(N_CLUSTERS):
        mat[i, c] = ((cluster_labels == c) & rmask).sum()
mat_pct = mat / mat.sum(axis=1, keepdims=True) * 100  # row-normalised %

fig, ax = plt.subplots(figsize=(max(6, N_CLUSTERS * 1.2), 4))
im = ax.imshow(mat_pct, aspect='auto', cmap='Blues')
ax.set_xticks(range(N_CLUSTERS))
ax.set_xticklabels([f'Cluster {c}' for c in range(N_CLUSTERS)], rotation=30, ha='right')
ax.set_yticks(range(len(region_names)))
ax.set_yticklabels(region_names)
for i in range(len(region_names)):
    for j in range(N_CLUSTERS):
        v = mat_pct[i, j]
        ax.text(j, i, f'{v:.0f}%', ha='center', va='center',
                fontsize=9, color='white' if v > 50 else '#222')
plt.colorbar(im, ax=ax, label='% of region voxels')
ax.set_title('Cluster composition by anatomical region (row-normalised)', fontsize=12)
plt.tight_layout(); plt.show()

## 8. Cross-Subject Consistency

Repeat clustering for a second subject and measure cluster centroid distance.

In [ ]:
from scipy.spatial.distance import cdist

# 对 local_subjects[1:] 中最多 4 个被试计算跨被试簇中心相似度
compare_subjects = local_subjects[1:5]
c1 = km.cluster_centers_   # Subject 1 的簇中心 (K, 32)

all_dists = []   # 每个对比被试的 nearest-cluster cosine distance
labels_compared = []

for subj_dir in compare_subjects:
    path2 = str(subj_dir / DIFF_SUBPATH) + '/'
    print(f'  Processing {subj_dir.name} ...', end=' ', flush=True)
    Generator.load_data(path=path2)

    ch2 = defaultdict(list)
    for _ in range(N_REPEATS):
        g = Generator.sample_graph(
            N=N_SAMPLE, k=config['k'], perplexity=config['perplexity'],
            sampling_method='random', gen_feature_mode=config['feature_mode'],
            sigma_method=config['sigma_method'],
        )
        h = extract_nofe(g)
        c = g.normed_coords.cpu().numpy()
        keys = (c * GRID).round().astype(int)
        for hh, kk in zip(h, keys):
            ch2[tuple(kk)].append(hh)
        del g, h, c, keys; _flush_mps()

    hidden2 = np.stack([np.mean(v, axis=0) for v in ch2.values()])
    X2      = StandardScaler().fit_transform(hidden2)
    km2     = KMeans(n_clusters=N_CLUSTERS, random_state=0, n_init=20).fit(X2)
    c2      = km2.cluster_centers_

    dist_mat = cdist(c1, c2, metric='cosine')
    min_dist = dist_mat.min(axis=1).mean()
    all_dists.append(min_dist)
    labels_compared.append(subj_dir.name)
    print(f'mean nearest-cluster dist = {min_dist:.4f}')
    del hidden2, X2, km2, c2, ch2; gc.collect(); _flush_mps()

# 汇总
print(f'\nCross-subject consistency (vs {SUBJECT.name}):')
for name, d in zip(labels_compared, all_dists):
    bar = '█' * int((1 - d) * 30)
    print(f'  {name}: cosine dist={d:.4f}  consistency={bar}')
print(f'\nMean cosine distance: {np.mean(all_dists):.4f}  (0=identical, 1=orthogonal)')

nofe feature + cluster vs default cluster